# Kaki — Model B: Transfer Risk (PaySim Random Forest)
Walkthrough of the Model B half of `train.py`.

**Caveats (also exported in `kaki_model.json`):** PaySim balances leak the label (fraud empties the
account), amounts are simulated units rather than SGD, fraud occurs only in TRANSFER/CASH_OUT, and PaySim has
no "recipient known" field — recipient familiarity is handled by the hand-written `kaki_ml.transfer_heuristic`,
never presented as learned.

In [ ]:
import os, sys, platform
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Machine:', platform.machine(), '| Python', platform.python_version())
# pip install -r requirements-train.txt   (run once in a terminal)
import kaki_ml, train

In [ ]:
raw, src = train.load_paysim(legit_sample=200_000)
print('source:', src)
raw.groupby('type').isFraud.agg(['count', 'sum'])

In [ ]:
rf, metrics = train.train_txn_model(raw)
{k: v for k, v in metrics.items() if k != 'caveat'}

In [ ]:
# Combined decision for one scenario: RF probability, heuristic floor, message context
feats = kaki_ml.txn_features(50000, 55000, 5000, 'TRANSFER', 22)
p_model = float(rf.predict_proba([feats])[0, 1])
floor = kaki_ml.transfer_heuristic(50000, 55000, recipient_known=False)
print(f'model={p_model:.2f} heuristic_floor={floor:.2f} -> risk={max(p_model, floor):.2f}')